In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



# Load data
data_frame= pd.read_csv('/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/FolicAcid_T2_Feb9/Network_outputs/Compiled_Networks.csv')  # Ensure to change the path as needed
data_frame.columns

In [ ]:
data_frame.columns

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from matplotlib.backends.backend_pdf import PdfPages

# ------------------------------
# Define the output folder and PDF file for saving plots.
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/FolicAcid_T2_Feb9/Network_outputs/plots/"  # UPDATE with your desired path
os.makedirs(output_folder, exist_ok=True)
pdf_filename = os.path.join(output_folder, "all_columns_category_div_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# ------------------------------
# Assume data_frame is your original DataFrame with columns such as:
# 'Run_ID', 'DIV', 'Well', 'NeuronType', 'Time', 'Chip_ID', ...,
# and the list–columns like 'Burst_Peak_List', 'Abs_Burst_Peak_List', 
# 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List'
# ------------------------------

# List of columns to process (all as originally stored in string format)
columns_to_process = [
    'Burst_Peak_List',
    'Abs_Burst_Peak_List',
    'Burst_Times_List',
    'IBI_List',
    'SpikesPerBurst_List'
]

# Define the grouping keys (unique category is defined by Chip_ID, NeuronType, and Well)
group_keys = ["Chip_ID", "NeuronType", "Well"]

# Loop over each column in columns_to_process
for column in columns_to_process:
    # Work on a copy so that modifications for one column don't affect the others.
    df_col = data_frame.copy()

    # Step 1: Convert the comma-separated strings into lists of floats.
    df_col[column] = df_col[column].apply(lambda x: [float(i) for i in x.split(",")])
    
    # Step 2: Explode the column so that each element becomes its own row.
    exploded_df = df_col.explode(column)
    
    # Rename the exploded column to "Value" and convert to numeric.
    exploded_df.rename(columns={column: "Value"}, inplace=True)
    exploded_df["Value"] = pd.to_numeric(exploded_df["Value"], errors="coerce")
    
    # Create a log-transformed version (using a shift to avoid log(0))
    exploded_df["LogValue"] = np.log10(exploded_df["Value"] + 1)
    
    # Step 3: Define bins for histogram calculation for both original and log–transformed data.
    bins = np.histogram_bin_edges(exploded_df["Value"], bins='auto')
    log_bins = np.histogram_bin_edges(exploded_df["LogValue"], bins='auto')
    
    # Prepare lists to collect the smoothed probability distributions.
    group_counts = []
    log_group_counts = []
    
    # Group by the unique combination (Chip_ID, NeuronType, Well) first.
    for cat, group in exploded_df.groupby(group_keys):
        # Within each category, further group by DIV.
        for div, sub_group in group.groupby("DIV"):
            # --- Original Values ---
            counts, _ = np.histogram(sub_group["Value"], bins=bins)
            if counts.sum() > 0:
                probabilities = counts / counts.sum()
            else:
                probabilities = counts  # avoid division by zero
            smoothed_probabilities = gaussian_filter1d(probabilities, sigma=1)
            for bin_left, prob in zip(bins[:-1], smoothed_probabilities):
                group_counts.append({
                    "Chip_ID": cat[0],
                    "NeuronType": cat[1],
                    "Well": cat[2],
                    "DIV": div,
                    "Bin_Start": bin_left,
                    "Probability": prob
                })
            
            # --- Log–Transformed Values ---
            log_counts, _ = np.histogram(sub_group["LogValue"], bins=log_bins)
            if log_counts.sum() > 0:
                log_probabilities = log_counts / log_counts.sum()
            else:
                log_probabilities = log_counts
            log_smoothed_probabilities = gaussian_filter1d(log_probabilities, sigma=1)
            for bin_left, prob in zip(log_bins[:-1], log_smoothed_probabilities):
                log_group_counts.append({
                    "Chip_ID": cat[0],
                    "NeuronType": cat[1],
                    "Well": cat[2],
                    "DIV": div,
                    "Bin_Start": bin_left,
                    "Probability": prob
                })
    
    # Convert the lists to DataFrames.
    smoothed_df = pd.DataFrame(group_counts)
    log_smoothed_df = pd.DataFrame(log_group_counts)
    
    # Get all unique categories (unique combination of Chip_ID, NeuronType, Well).
    unique_categories = smoothed_df[group_keys].drop_duplicates()
    
    # --- FIGURE 1: Original Data Distributions ---
    n_categories = len(unique_categories)
    ncols = 2  # you can adjust the number of columns in the subplot grid
    nrows = int(np.ceil(n_categories / ncols))
    
    fig_orig, axes_orig = plt.subplots(nrows, ncols, figsize=(15, 6 * nrows), sharex=True, sharey=True)
    # Flatten axes array for easy iteration.
    if n_categories > 1:
        axes_orig = axes_orig.flatten()
    else:
        axes_orig = [axes_orig]
    
    # Loop over each unique category and plot its distribution.
    for i, (_, cat_row) in enumerate(unique_categories.iterrows()):
        chip_id = cat_row["Chip_ID"]
        neuron_type = cat_row["NeuronType"]
        well = cat_row["Well"]
        
        subset = smoothed_df[
            (smoothed_df["Chip_ID"] == chip_id) &
            (smoothed_df["NeuronType"] == neuron_type) &
            (smoothed_df["Well"] == well)
        ]
        
        ax = axes_orig[i]
        sns.lineplot(data=subset, x="Bin_Start", y="Probability", hue="DIV", marker="o", ax=ax)
        ax.set_title(f"{chip_id} | {neuron_type} | {well}", fontsize=10)
        ax.set_xlabel("Value Range")
        ax.set_ylabel("Probability")
        ax.grid(True)
    
    # Remove any unused subplots.
    for j in range(i + 1, len(axes_orig)):
        fig_orig.delaxes(axes_orig[j])
    
    fig_orig.suptitle(f"{column} Distribution (Original)", fontsize=16, y=1.02)
    fig_orig.tight_layout()
    
    # Save the original figure and add it to the PDF.
    orig_filename = os.path.join(output_folder, f"{column}_distribution_original.png")
    fig_orig.savefig(orig_filename, dpi=300)
    pdf_pages.savefig(fig_orig)
    plt.show()
    plt.close(fig_orig)
    
    # --- FIGURE 2: Log–Transformed Data Distributions ---
    fig_log, axes_log = plt.subplots(nrows, ncols, figsize=(15, 6 * nrows), sharex=True, sharey=True)
    if n_categories > 1:
        axes_log = axes_log.flatten()
    else:
        axes_log = [axes_log]
    
    for i, (_, cat_row) in enumerate(unique_categories.iterrows()):
        chip_id = cat_row["Chip_ID"]
        neuron_type = cat_row["NeuronType"]
        well = cat_row["Well"]
        
        subset = log_smoothed_df[
            (log_smoothed_df["Chip_ID"] == chip_id) &
            (log_smoothed_df["NeuronType"] == neuron_type) &
            (log_smoothed_df["Well"] == well)
        ]
        
        ax = axes_log[i]
        sns.lineplot(data=subset, x="Bin_Start", y="Probability", hue="DIV", marker="o", ax=ax)
        ax.set_title(f"{chip_id} | {neuron_type} | {well}", fontsize=10)
        ax.set_xlabel("Log(Value Range)")
        ax.set_ylabel("Probability")
        ax.grid(True)
    
    for j in range(i + 1, len(axes_log)):
        fig_log.delaxes(axes_log[j])
    
    fig_log.suptitle(f"{column} Distribution (Log-Transformed)", fontsize=16, y=1.02)
    fig_log.tight_layout()
    
    log_filename = os.path.join(output_folder, f"{column}_distribution_log.png")
    fig_log.savefig(log_filename, dpi=300)
    pdf_pages.savefig(fig_log)
    plt.show()
    plt.close(fig_log)

# Close the PDF file once all figures have been added.
pdf_pages.close()

In [ ]:
from scipy.ndimage import gaussian_filter1d
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.backends.backend_pdf import PdfPages

# ------------------------------
# USER-DEFINED OPTIONS
# ------------------------------
# Set up the output folder (update the path as needed)
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/FolicAcid_T2_Feb9/Network_outputs/plots/"
os.makedirs(output_folder, exist_ok=True)

# Set up the PDF file for saving plots
pdf_filename = os.path.join(output_folder, "all_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# ------------------------------
# Columns to process (original string columns with comma-separated values)
# ------------------------------
columns_to_process = [
    'Burst_Peak_List',
    'Abs_Burst_Peak_List',
    'Burst_Times_List',
    'IBI_List',
    'SpikesPerBurst_List'
]

# Loop over each column in columns_to_process
for column in columns_to_process:
    # Work on a temporary copy so the original data_frame remains unaltered.
    temp_df = data_frame.copy()

    # Step 1: Convert the column strings into lists of floats.
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

    # Step 2: Explode the lists into individual rows.
    exploded_df = temp_df.explode(column)

    # Rename the exploded column to "Value" and convert to numeric.
    exploded_df.rename(columns={column: "Value"}, inplace=True)
    exploded_df["Value"] = pd.to_numeric(exploded_df["Value"], errors="coerce")

    # Also create a log-transformed version (using +1 to avoid log(0)).
    exploded_df["LogValue"] = np.log10(exploded_df["Value"] + 1)

    # Step 3: Define bins automatically for histogram computation.
    bins = np.histogram_bin_edges(exploded_df["Value"], bins='auto')
    log_bins = np.histogram_bin_edges(exploded_df["LogValue"], bins='auto')
    
    # Step 4: Compute histogram counts, probabilities, and apply Gaussian smoothing.
    group_counts = []
    log_group_counts = []

    # Group by the unique combination: DIV, Well, and NeuronType.
    for (div, well, neuron_type), group in exploded_df.groupby(["DIV", "Well", "NeuronType"]):
        # --- For Original Values ---
        counts, _ = np.histogram(group["Value"], bins=bins)
        if counts.sum() > 0:
            probabilities = counts / counts.sum()
        else:
            probabilities = counts  # avoid division by zero
        smoothed_probabilities = gaussian_filter1d(probabilities, sigma=1)

        for bin_left, prob in zip(bins[:-1], smoothed_probabilities):
            group_counts.append({
                "DIV": div,
                "Well": well,
                "NeuronType": neuron_type,
                "Bin_Start": bin_left,
                "Probability": prob
            })

        # --- For Log-Transformed Values ---
        log_counts, _ = np.histogram(group["LogValue"], bins=log_bins)
        if log_counts.sum() > 0:
            log_probabilities = log_counts / log_counts.sum()
        else:
            log_probabilities = log_counts
        log_smoothed_probabilities = gaussian_filter1d(log_probabilities, sigma=1)

        for bin_left, prob in zip(log_bins[:-1], log_smoothed_probabilities):
            log_group_counts.append({
                "DIV": div,
                "Well": well,
                "NeuronType": neuron_type,
                "Bin_Start": bin_left,
                "Probability": prob
            })

    # Convert the lists of dictionaries to DataFrames.
    smoothed_df = pd.DataFrame(group_counts)
    log_smoothed_df = pd.DataFrame(log_group_counts)

    # Step 5: Create a custom palette.
    # We want to assign shades of blue for as many wells that are "0FA" 
    # and shades of red for as many wells that are "10FA".
    # Get the unique wells for each NeuronType.
    unique_0fa_wells = sorted(smoothed_df.loc[smoothed_df["NeuronType"] == "0FA", "Well"].unique())
    unique_10fa_wells = sorted(smoothed_df.loc[smoothed_df["NeuronType"] == "10FA", "Well"].unique())

    # Generate palettes with as many colors as there are unique wells.
    blue_palette = sns.color_palette("Blues", len(unique_0fa_wells))
    red_palette = sns.color_palette("Reds", len(unique_10fa_wells))

    # Build a dictionary mapping each (Well, NeuronType) combination to a color.
    palette = {}
    for i, well in enumerate(unique_0fa_wells):
        palette[(well, "0FA")] = blue_palette[i]
    for i, well in enumerate(unique_10fa_wells):
        palette[(well, "10FA")] = red_palette[i]

    # Step 6: Create subplots for each DIV for the Original data distributions.
    divs = smoothed_df["DIV"].unique()
    num_rows, num_cols = (len(divs) + 1) // 2, 2  # Adjust rows/columns as needed.
    fig_orig, axes_orig = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)
    axes_orig = axes_orig.flatten() if len(divs) > 1 else [axes_orig]

    for i, div in enumerate(divs):
        ax = axes_orig[i]
        subset = smoothed_df[smoothed_df["DIV"] == div]
        # Group further by (Well, NeuronType) so that each unique combination gets its own line.
        for (well, nt), well_data in subset.groupby(["Well", "NeuronType"]):
            # Get the color from the palette based on (well, neuron_type).
            color = palette.get((well, nt), "gray")
            sns.lineplot(
                data=well_data,
                x="Bin_Start",
                y="Probability",
                color=color,
                label=f"Well {well} - {nt}",
                marker="o",
                ax=ax
            )
        ax.set_title(f"DIV {div} - Smoothed Distribution", fontsize=10)
        ax.set_xlabel("Value Range")
        ax.set_ylabel("Probability")
        ax.legend(title="Category", fontsize=8, loc="upper right")
        ax.grid()

    # Remove any unused subplots.
    for j in range(i + 1, len(axes_orig)):
        fig_orig.delaxes(axes_orig[j])

    fig_orig.suptitle(f"{column} Distribution (Original)", fontsize=16, y=1.02)
    plt.tight_layout()
    # Save the original figure to PDF.
    pdf_pages.savefig(fig_orig)
    plt.show()
    plt.close(fig_orig)

    # Step 7: Create subplots for the Log–Transformed data distributions.
    divs_log = log_smoothed_df["DIV"].unique()
    num_rows_log, num_cols_log = (len(divs_log) + 1) // 2, 2
    fig_log, axes_log = plt.subplots(num_rows_log, num_cols_log, figsize=(15, 6 * num_rows_log), sharex=True, sharey=True)
    axes_log = axes_log.flatten() if len(divs_log) > 1 else [axes_log]

    for i, div in enumerate(divs_log):
        ax = axes_log[i]
        subset = log_smoothed_df[log_smoothed_df["DIV"] == div]
        for (well, nt), well_data in subset.groupby(["Well", "NeuronType"]):
            color = palette.get((well, nt), "gray")
            sns.lineplot(
                data=well_data,
                x="Bin_Start",
                y="Probability",
                color=color,
                label=f"Well {well} - {nt}",
                marker="o",
                ax=ax
            )
        ax.set_title(f"DIV {div} - Smoothed Distribution (Log)", fontsize=10)
        ax.set_xlabel("Log(Value Range)")
        ax.set_ylabel("Probability")
        ax.legend(title="Category", fontsize=8, loc="upper right")
        ax.grid()

    for j in range(i + 1, len(axes_log)):
        fig_log.delaxes(axes_log[j])

    fig_log.suptitle(f"{column} Distribution (Log-Transformed)", fontsize=16, y=1.02)
    plt.tight_layout()
    # Save the log-transformed figure to PDF.
    pdf_pages.savefig(fig_log)
    plt.show()
    plt.close(fig_log)

# After processing all columns, close the PDF file.
pdf_pages.close()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# ------------------------------
# USER-DEFINED OPTIONS
# ------------------------------
# Define output folder (adjust to your actual path)
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/FolicAcid_T2_Feb9/Network_outputs/plots/"
os.makedirs(output_folder, exist_ok=True)

# Define a PDF file to collect all plots
pdf_filename = os.path.join(output_folder, "boxplots_scatter.pdf")
pdf_pages = PdfPages(pdf_filename)

# ------------------------------
# Columns to process (original string columns with comma-separated values)
# ------------------------------
columns_to_process = [
    'Burst_Peak_List',
    'Abs_Burst_Peak_List',
    'Burst_Times_List',
    'IBI_List',
    'SpikesPerBurst_List'
]

# Loop over each column in columns_to_process
for column in columns_to_process:
    # Step 1: Convert column strings to lists of floats without altering the original dataframe
    temp_df = data_frame.copy()
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])
    
    # Step 2: Explode the lists into individual rows
    exploded_df = temp_df.explode(column)
    
    # Rename the exploded column to "Value" and convert to numeric
    exploded_df.rename(columns={column: "Value"}, inplace=True)
    exploded_df["Value"] = pd.to_numeric(exploded_df["Value"], errors="coerce")
    
    # Create a new column "Category" that combines Well and NeuronType
    exploded_df["Category"] = exploded_df.apply(lambda row: f"Well {row['Well']} - {row['NeuronType']}", axis=1)
    
    # Step 3: Create box plots with overlayed scatter points
    divs = exploded_df["DIV"].unique()
    num_rows, num_cols = (len(divs) + 1) // 2, 2  # Adjust rows/columns based on DIV count
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 6 * num_rows), sharex=True, sharey=True)
    axes = axes.flatten() if len(divs) > 1 else [axes]
    
    # Step 4: Create a custom palette for the scatter and box plots.
    # We want to assign shades of blue for wells with NeuronType "0FA" and
    # shades of red for wells with NeuronType "10FA".
    
    # Determine unique wells for each NeuronType from the exploded data
    unique_0fa_wells = sorted(exploded_df.loc[exploded_df["NeuronType"] == "0FA", "Well"].unique())
    unique_10fa_wells = sorted(exploded_df.loc[exploded_df["NeuronType"] == "10FA", "Well"].unique())
    
    # Generate color palettes: one for each set of unique wells.
    blue_palette = sns.color_palette("Blues", len(unique_0fa_wells))
    red_palette = sns.color_palette("Reds", len(unique_10fa_wells))
    
    # Build a dictionary mapping each (Well, NeuronType) combination to a color.
    palette = {}
    for i, well in enumerate(unique_0fa_wells):
        palette[(well, "0FA")] = blue_palette[i]
    for i, well in enumerate(unique_10fa_wells):
        palette[(well, "10FA")] = red_palette[i]
    
    # Create a palette mapping for the "Category" strings.
    unique_categories = exploded_df["Category"].unique()
    cat_palette = {}
    for cat in unique_categories:
        # Expected format: "Well {well} - {NeuronType}"
        parts = cat.split(" - ")
        well_str = parts[0].replace("Well ", "")
        try:
            well = int(well_str)
        except:
            well = well_str  # fallback if conversion fails
        nt = parts[1]
        cat_palette[cat] = palette.get((well, nt), "gray")
    
    # Step 5: Plot box plots and overlay scatter points for each DIV.
    for i, div in enumerate(divs):
        ax = axes[i]
    
        # Subset data for current DIV
        div_data = exploded_df[exploded_df["DIV"] == div]
    
        # Plot box plot: x-axis is the composite "Category", y-axis is "Value"
        sns.boxplot(
            data=div_data,
            x="Category",
            y="Value",
            ax=ax,
            palette=cat_palette,
            width=0.6,
            showcaps=True,
            showmeans=False,
            boxprops=dict(alpha=0.7)
        )
    
        # Overlay scatter points (using the same category and palette)
        sns.stripplot(
            data=div_data,
            x="Category",
            y="Value",
            hue="Category",
            palette=cat_palette,
            dodge=False,  # Align with boxplot positions
            size=5,
            alpha=0.8,
            jitter=True,
            ax=ax
        )
    
        # Customize plot appearance
        ax.set_title(f"DIV {div} - Box Plot with Overlayed Scatter", fontsize=10)
        ax.set_ylabel("Value")
        ax.set_xlabel("")
        ax.grid(axis="y")
    
        # Remove duplicate legend entries (stripplot adds a legend)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            # Show only one legend entry per category
            ax.legend(handles[0:len(cat_palette)], labels[0:len(cat_palette)], title="Category", fontsize=8, loc="upper right", frameon=False)
    
    # Remove any unused subplots if the number of DIVs is less than the grid size.
    for j in range(len(divs), len(axes)):
        fig.delaxes(axes[j])
    
    # Step 6: Adjust layout and add a super title that indicates which column is being plotted.
    plt.tight_layout()
    plt.suptitle(f"Box Plot with Scatter Overlay of {column} Across DIVs\n(Categorized by Well & NeuronType)", fontsize=16, y=1.02)
    
    # Save the figure to the PDF
    pdf_pages.savefig(fig)
    plt.show()
    plt.close(fig)

# Close the PDF file after all plots are generated
pdf_pages.close()

In [ ]:
from itertools import combinations
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd

# --- User-defined paths and settings ---
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/FolicAcid_T2_Feb9/Network_outputs/plots/"
os.makedirs(output_folder, exist_ok=True)
pdf_filename = os.path.join(output_folder, "pair_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# --- Columns to process ---
columns_to_process = [
    'Burst_Peak_List',
    'Burst_Times_List',
    'IBI_List',
    'SpikesPerBurst_List',
    'Abs_Burst_Peak_List'
]

# --- Step 1: Convert the specified columns into lists of floats ---
temp_df = data_frame.copy()
for column in columns_to_process:
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])

# --- Truncate lists to match the shortest length for each row ---
def truncate_lists(row, columns):
    min_length = min(len(row[col]) for col in columns)
    for col in columns:
        row[col] = row[col][:min_length]
    return row

temp_df = temp_df.apply(truncate_lists, columns=columns_to_process, axis=1)

# --- Generate all pairwise combinations of the columns ---
column_combinations = list(combinations(columns_to_process, 2))

# --- Build a custom palette based on composite groups (Chip_ID, Well, NeuronType) ---
# First, get the unique composite groups from the full data.
# (Assuming these columns exist in data_frame.)
composite_df = temp_df[['Chip_ID', 'Well', 'NeuronType']].drop_duplicates()

# For groups with NeuronType "0FA"
unique_0fa = composite_df[composite_df["NeuronType"] == "0FA"]
unique_0fa_keys = list(unique_0fa.apply(lambda row: (row["Chip_ID"], row["Well"], row["NeuronType"]), axis=1))
# For groups with NeuronType "10FA"
unique_10fa = composite_df[composite_df["NeuronType"] == "10FA"]
unique_10fa_keys = list(unique_10fa.apply(lambda row: (row["Chip_ID"], row["Well"], row["NeuronType"]), axis=1))

# Generate color palettes: one for each set.
blue_palette = plt.cm.Blues(np.linspace(0.3, 0.8, len(unique_0fa_keys)))  # shades of blue
red_palette = plt.cm.Reds(np.linspace(0.3, 0.8, len(unique_10fa_keys)))    # shades of red

# Build a dictionary mapping each composite key to its color.
palette = {}
for i, key in enumerate(unique_0fa_keys):
    palette[key] = blue_palette[i]
for i, key in enumerate(unique_10fa_keys):
    palette[key] = red_palette[i]

# --- Get unique DIV values ---
divs = temp_df['DIV'].unique()

# --- Loop over each DIV to generate plots ---
for div in divs:
    subset = temp_df[temp_df['DIV'] == div]
    
    # Prepare a subplot grid for the current DIV; one row per column pair.
    num_plots = len(column_combinations)
    fig, axes = plt.subplots(num_plots, 1, figsize=(8, 6 * num_plots))
    if num_plots == 1:
        axes = [axes]
    fig.suptitle(f"Scatter Plots for DIV {div}", fontsize=16, y=0.92)
    
    # Loop over each column combination and corresponding axis.
    for (col_x, col_y), ax in zip(column_combinations, axes):
        ax.set_title(f"{col_x} vs {col_y}", fontsize=12)
        ax.set_xlabel(col_x)
        ax.set_ylabel(col_y)
        
        # Group by composite key: (Chip_ID, Well, NeuronType)
        for key, group_data in subset.groupby(["Chip_ID", "Well", "NeuronType"]):
            # key is a tuple: (Chip_ID, Well, NeuronType)
            # Explode the lists for the current pair of columns
            x_values = group_data[col_x].explode().astype(float)
            y_values = group_data[col_y].explode().astype(float)
            
            # Calculate Pearson correlation coefficient (r)
            if len(x_values) > 1 and len(y_values) > 1:
                r = np.corrcoef(x_values, y_values)[0, 1]
                # Compute Fisher z-transformation if |r| < 1
                if abs(r) < 1:
                    z_value = 0.5 * np.log((1 + r) / (1 - r))
                else:
                    z_value = np.nan
            else:
                r = np.nan
                z_value = np.nan
            
            # Get the assigned color from the palette for this composite key
            color = palette.get(key, "gray")
            
            # Build a label that shows composite info plus correlation scores.
            chip, well, nt = key
            label = f"Chip {chip} - Well {well} - {nt} (r={r:.2f}, z={z_value:.2f})"
            
            # Plot scatter points.
            ax.scatter(x_values, y_values, label=label, color=color, alpha=0.8)
        
        # Show legend (you might limit number of legend entries if too many)
        ax.legend(loc='upper left', fontsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    # Save individual plot as PNG.
    png_filename = os.path.join(output_folder, f"scatter_div_{div}.png")
    plt.savefig(png_filename, dpi=300)
    
    # Save the figure to the PDF.
    pdf_pages.savefig(fig)
    plt.show()
    plt.close(fig)

# Close the PDF file after all plots are generated.
pdf_pages.close()

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

# ------------------------------
# USER-DEFINED SETTINGS
# ------------------------------
# Define output folder (adjust to your actual path)
output_folder = "/mnt/disk15tb/paula/Main_DA_Projects/data_analysis_output/Primary Neurons/FolicAcid_T2_Feb9/Network_outputs/plots/"
os.makedirs(output_folder, exist_ok=True)

# Create a single PDF for all scatter plots
pdf_filename = os.path.join(output_folder, "log_transformed_scatter_plots.pdf")
pdf_pages = PdfPages(pdf_filename)

# ------------------------------
# Columns to process (list-columns)
# ------------------------------
columns_to_process = [
    'Burst_Peak_List',
    'Burst_Times_List',
    'IBI_List',
    'SpikesPerBurst_List',
    'Abs_Burst_Peak_List'
]

# --- Step 1: Convert column strings to lists of floats and apply log transformation ---
temp_df = data_frame.copy()
for column in columns_to_process:
    # Convert comma–separated strings to an array of floats, add 1 to avoid log(0), then apply log10.
    temp_df[column] = temp_df[column].apply(lambda x: np.log10(np.array([float(i) for i in x.split(",")]) + 1))

# --- Step 2: Truncate lists to match the shortest length in each row ---
def truncate_lists(row, columns):
    min_length = min(len(row[col]) for col in columns)
    for col in columns:
        row[col] = row[col][:min_length]
    return row

temp_df = temp_df.apply(truncate_lists, columns=columns_to_process, axis=1)

# --- Step 3: Generate all pairwise combinations of the columns ---
column_combinations = list(combinations(columns_to_process, 2))

# --- Step 4: Build composite grouping palette ---
# We want to group by (Chip_ID, Well, NeuronType) and assign:
# - For groups with NeuronType "0FA": distinct blue shades.
# - For groups with NeuronType "10FA": distinct red shades.
composite_df = temp_df[['Chip_ID', 'Well', 'NeuronType']].drop_duplicates()

# Get unique composite keys for each NeuronType.
unique_0fa = composite_df[composite_df["NeuronType"] == "0FA"]
unique_0fa_keys = list(unique_0fa.apply(lambda row: (row["Chip_ID"], row["Well"], row["NeuronType"]), axis=1))
unique_10fa = composite_df[composite_df["NeuronType"] == "10FA"]
unique_10fa_keys = list(unique_10fa.apply(lambda row: (row["Chip_ID"], row["Well"], row["NeuronType"]), axis=1))

# Generate color palettes using matplotlib’s colormap.
blue_palette = plt.cm.Blues(np.linspace(0.3, 0.8, len(unique_0fa_keys)))  # shades of blue
red_palette = plt.cm.Reds(np.linspace(0.3, 0.8, len(unique_10fa_keys)))    # shades of red

# Build a dictionary mapping each composite key to its assigned color.
palette = {}
for i, key in enumerate(unique_0fa_keys):
    palette[key] = blue_palette[i]
for i, key in enumerate(unique_10fa_keys):
    palette[key] = red_palette[i]

# --- Step 5: Get unique DIV values ---
divs = temp_df['DIV'].unique()

# --- Step 6: Loop over each DIV and generate log–transformed pair scatter plots ---
for div in divs:
    subset = temp_df[temp_df['DIV'] == div]
    
    # Prepare a subplot grid: one row per column combination.
    num_plots = len(column_combinations)
    fig, axes = plt.subplots(num_plots, 1, figsize=(8, 6 * num_plots))
    if num_plots == 1:
        axes = [axes]
    fig.suptitle(f"Log-Transformed Scatter Plots for DIV {div}", fontsize=16, y=0.92)
    
    # Loop over each pair of columns and corresponding axis.
    for (col_x, col_y), ax in zip(column_combinations, axes):
        ax.set_title(f"log({col_x}) vs log({col_y})", fontsize=12)
        ax.set_xlabel(f"log({col_x})")
        ax.set_ylabel(f"log({col_y})")
        
        # Group by composite key: (Chip_ID, Well, NeuronType)
        for key, group_data in subset.groupby(["Chip_ID", "Well", "NeuronType"]):
            # Explode the lists for the current pair of columns.
            x_values = group_data[col_x].explode().astype(float)
            y_values = group_data[col_y].explode().astype(float)
            
            # Calculate Pearson correlation coefficient (r) and Fisher z–transformation.
            if len(x_values) > 1 and len(y_values) > 1:
                r = np.corrcoef(x_values, y_values)[0, 1]
                if abs(r) < 1:
                    z_value = 0.5 * np.log((1 + r) / (1 - r))
                else:
                    z_value = np.nan
            else:
                r = np.nan
                z_value = np.nan
            
            # Retrieve the color for this composite group.
            color = palette.get(key, "gray")
            chip, well, nt = key
            label = f"Chip {chip} - Well {well} - {nt} (r={r:.2f}, z={z_value:.2f})"
            
            ax.scatter(x_values, y_values, label=label, color=color, alpha=0.8)
        
        ax.legend(loc='upper left', fontsize=8)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    # Save the individual DIV plot as PNG.
    png_filename = os.path.join(output_folder, f"log_scatter_pairwise_div_{div}.png")
    plt.savefig(png_filename, dpi=300)
    
    # Save the figure to the PDF.
    pdf_pages.savefig(fig)
    plt.show()
    plt.close(fig)

# --- Step 7: Close the PDF file after all plots are generated ---
pdf_pages.close()

In [ ]:
# Columns to process
columns_to_process = ['Burst_Peak_List', 'Abs_Burst_Peak_List', 'Burst_Times_List', 'IBI_List', 'SpikesPerBurst_List']

# Step 1: Convert the specified columns into individual lists of floats
temp_df = df.copy()
for column in columns_to_process:
    temp_df[column] = temp_df[column].apply(lambda x: [float(i) for i in x.split(",")])
    print(len(temp_df[column]))



# Step 2: Explode all specified columns into individual rows
exploded_dataframes = []
for column in columns_to_process:
    exploded_df = temp_df.explode(column)
    exploded_df.rename(columns={column: "Value"}, inplace=True)
    exploded_df["Feature"] = column  # Add a column to indicate the feature name
    exploded_dataframes.append(exploded_df[["DIV", "Well", "Feature", "Value"]])

# Combine all the exploded DataFrames
combined_df = pd.concat(exploded_dataframes, axis=0)

# Get unique DIV values
unique_divs = combined_df["DIV"].unique()

# Get all unique features
unique_features = combined_df["Feature"].unique()
from itertools import combinations
# Generate all possible pairs of features
feature_pairs = list(combinations(unique_features, 2))
# Loop through each DIV
for div_value in unique_divs:
    # Filter data for this DIV
    div_df = combined_df[combined_df["DIV"] == div_value]
    
    # Pivot the table so that features become columns
    pivoted = div_df.pivot(index="Well", columns="Feature", values="Value")
    
    # Loop through each pair of features
    for feature_x, feature_y in feature_pairs:
        # Check if both features exist in the pivoted data
        if feature_x in pivoted.columns and feature_y in pivoted.columns:
            # Drop rows where the selected features aren't both present
            pivoted_clean = pivoted.dropna(subset=[feature_x, feature_y])
            
            # Plotting
            plt.figure(figsize=(8, 6))
            
            # Plot each well in a different color
            for well in pivoted_clean.index:
                plt.scatter(
                    pivoted_clean.loc[well, feature_x],
                    pivoted_clean.loc[well, feature_y],
                    label=f"Well {well}",
                    alpha=0.7
                )
            
            plt.xlabel(feature_x)
            plt.ylabel(feature_y)
            plt.title(f"DIV {div_value} - {feature_x} vs {feature_y}")
            plt.legend(title="Well")
            plt.grid(True)
            plt.show()